## Test the chain: Raw -> Enriched -> ArbFree -> OptionFilter


In [1]:

%load_ext autoreload
%autoreload 1

%aimport openqai.core
%aimport openqai.core.biz_calendar
%aimport openqai.data.widb
%aimport openqai.data.widb.day
%aimport openqai.data.widb.arbfree
%aimport openqai.data.widb.options_filter

from datetime import date
from openqai.data.widb.day import collect_raw_day, enrich_day, ExpiryFilter, RawDayFilter
from openqai.data.widb.arbfree import mk_arb_free_day, MkArbFree
from openqai.data.widb.options_filter import OptionsFilter, filter_options_day
from openqai.core.biz_calendar import next_business_day
from cdxcore.verbose import Context
from IPython.display import display, Markdown


def test():
    ticker = "SPX"
    d      = next_business_day( date(2020, 1, 28), count_start=True )

    raw_day_filter = RawDayFilter()
    expiry_filter = ExpiryFilter()
    verbose = Context.all

    display(Markdown("## collect_raw_day"))
    raw_day = collect_raw_day(ticker, d, raw_day_filter=raw_day_filter, verbose=verbose, override_cache_mode="off" )
    display(raw_day)

    display(Markdown("## enrich_day"))
    enr_day = enrich_day(ticker, d, raw_day, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, override_cache_mode="off" )
    display(enr_day)    
    display(enr_day[['nms','nms_atm']])

    display(Markdown("## mk_arb_free_day"))
    mk_arb_free = MkArbFree()
    arb_free = mk_arb_free_day(ticker, d, enr_day, mk_arb_free=mk_arb_free, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose, override_cache_mode="off")
    display(arb_free)

    display(Markdown("## filter_options_day"))
    options_filter = OptionsFilter()
    opt_flt = filter_options_day( ticker, d, arb_free, options_filter = options_filter, mk_arb_free=mk_arb_free, raw_day_filter=raw_day_filter, expiry_filter=expiry_filter, verbose=verbose , override_cache_mode="off")
    display(opt_flt)
    return raw_day, enr_day, arb_free, opt_flt

raw_day, enr_day, arb_free, opt_flt = test()

## collect_raw_day

00: Collecting raw market data for SPX on 2020-01-28.
00: SPX 2020-01-28: collected 7304/16070 options. Removed: 731/16070 options with bid <= 0.011.


,spot,spot_prev,date,ticker,call_symbol,put_symbol,call_opt_id,put_opt_id,is_american,is_am,...,mkt_put_bid,mkt_call_ask,mkt_put_ask,mkt_call_log_volume,mkt_put_log_volume,mkt_call_log_open_interest,mkt_put_log_open_interest,expiry_id,bdte,tte_y
0,3276.23999,3243.629883,2020-01-28,SPX,SPXW 200129C3035000,SPXW 200129P3035000,131849080,131849198,False,False,...,0.050000,244.300003,0.1,0.000000,1.982271,0.000000,2.717670,202001291,1.000000,0.003968
1,3276.23999,3243.629883,2020-01-28,SPX,SPXW 200129C3045000,SPXW 200129P3045000,131849082,131849200,False,False,...,0.050000,234.300003,0.1,0.000000,2.206826,0.000000,3.122871,202001291,1.000000,0.003968
2,3276.23999,3243.629883,2020-01-28,SPX,SPXW 200129C3050000,SPXW 200129P3050000,131849083,131849201,False,False,...,0.050000,229.300003,0.1,0.000000,3.217221,0.477121,3.738225,202001291,1.000000,0.003968
3,3276.23999,3243.629883,2020-01-28,SPX,SPXW 200129C3055000,SPXW 200129P3055000,131849084,131849202,False,False,...,0.050000,224.300003,0.1,0.000000,2.363612,0.000000,2.369216,202001291,1.000000,0.003968
4,3276.23999,3243.629883,2020-01-28,SPX,SPXW 200129C3060000,SPXW 200129P3060000,131849085,131849203,False,False,...,0.050000,219.300003,0.1,0.000000,2.635484,0.000000,3.054613,202001291,1.000000,0.003968
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7299,3276.23999,3243.629883,2020-01-28,SPX,SPX 221216C4400000,SPX 221216P4400000,131820331,131820428,False,True,...,1098.800049,30.299999,1121.0,0.000000,0.000000,1.732394,0.000000,202212160,728.200012,2.889683
7300,3276.23999,3243.629883,2020-01-28,SPX,SPX 221216C4500000,SPX 221216P4500000,131820332,131820429,False,True,...,1187.900024,23.000000,1209.5,0.477121,0.000000,2.463893,0.000000,202212160,728.200012,2.889683
7301,3276.23999,3243.629883,2020-01-28,SPX,SPX 221216C4600000,SPX 221216P4600000,131820333,131820430,False,True,...,1278.199951,19.500000,1299.5,0.954243,0.000000,1.732394,0.000000,202212160,728.200012,2.889683
7302,3276.23999,3243.629883,2020-01-28,SPX,SPX 221216C4700000,SPX 221216P4700000,131965166,131965167,False,True,...,1369.400024,16.100000,1390.0,0.000000,0.000000,1.838849,0.000000,202212160,728.200012,2.889683


## enrich_day

00: SPX 2020-01-28: Enriched 5091 of 7304 raw options across 32 of 32 input expiries.
01:   Dropped the following 13 of 45 input expiries: 2020-03-04pm (insufficient traded strikes), 2020-05-15pm (insufficient traded strikes), 2020-06-19pm (insufficient traded strikes), 2020-06-30pm (insufficient traded strikes), 2020-07-17am (insufficient traded strikes), 2020-09-30pm (insufficient traded strikes), 2020-10-16am (insufficient traded strikes), 2020-11-20am (insufficient traded strikes), 2020-12-31pm (insufficient traded strikes), 2021-03-19am (insufficient traded strikes), 2021-06-18am (insufficient traded strikes), 2021-12-17am (insufficient traded strikes post filtering) and 2022-12-16am (insufficient traded strikes). 
01:   Dropped 552 strikes among the remaining 32 expiries: 2020-01-29pm (Dropped 6 strikes with p/c issues: 56/62 strikes left), 2020-01-31pm (Dropped 5 strikes with p/c issues: 120/125 strikes left), 2020-02-03pm (Dropped 7 strikes with p/c issues: 109/116 strikes left

,spot,spot_prev,date,ticker,call_symbol,put_symbol,call_opt_id,put_opt_id,is_american,is_am,...,pure_ask_delta,pure_bid_delta,pure_ask_gamma,pure_bid_gamma,pure_bid_vega,pure_ask_vega,pure_bid_theta,pure_ask_theta,pure_ask_dk,pure_bid_dk
0,3276.23999,3243.629883,2020-01-28,SPX,SPXW 200129C3035000,SPXW 200129P3035000,131849080,131849198,False,False,...,0.996542,0.998025,0.362582,0.233400,0.000395,0.000655,0.021201,0.037555,-0.996233,-0.997851
1,3276.23999,3243.629883,2020-01-28,SPX,SPXW 200129C3045000,SPXW 200129P3045000,131849082,131849200,False,False,...,0.996415,0.997955,0.389590,0.250749,0.000407,0.000676,0.021027,0.037298,-0.996108,-0.997782
2,3276.23999,3243.629883,2020-01-28,SPX,SPXW 200129C3050000,SPXW 200129P3050000,131849083,131849201,False,False,...,0.996348,0.997918,0.404277,0.260198,0.000414,0.000688,0.020939,0.037166,-0.996042,-0.997746
3,3276.23999,3243.629883,2020-01-28,SPX,SPXW 200129C3055000,SPXW 200129P3055000,131849084,131849202,False,False,...,0.996277,0.997879,0.419838,0.270221,0.000421,0.000700,0.020850,0.037032,-0.995973,-0.997708
4,3276.23999,3243.629883,2020-01-28,SPX,SPXW 200129C3060000,SPXW 200129P3060000,131849085,131849203,False,False,...,0.996204,0.997839,0.436347,0.280866,0.000429,0.000712,0.020761,0.036895,-0.995901,-0.997668
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6856,3276.23999,3243.629883,2020-01-28,SPX,SPX 210115C3650000,SPX 210115P3650000,131694745,131694747,False,True,...,0.180770,0.178180,2.364540,2.365833,0.256661,0.259000,0.014826,0.015105,-0.152987,-0.150913
6857,3276.23999,3243.629883,2020-01-28,SPX,SPX 210115C3675000,SPX 210115P3675000,132061008,132061009,False,True,...,0.160759,0.158244,2.228973,2.226851,0.237791,0.240258,0.013520,0.013789,-0.135483,-0.133461
6858,3276.23999,3243.629883,2020-01-28,SPX,SPX 210115C3700000,SPX 210115P3700000,131475177,131475271,False,True,...,0.142686,0.139961,2.085779,2.079993,0.219063,0.221946,0.012284,0.012575,-0.119713,-0.117518
6859,3276.23999,3243.629883,2020-01-28,SPX,SPX 210115C3750000,SPX 210115P3750000,132029666,132029667,False,True,...,0.111245,0.108499,1.790640,1.778205,0.183284,0.186602,0.010059,0.010354,-0.092454,-0.090244


... (2 outputs left out)

00: SPX 2020-01-28: Generating arb-free price surface.
01:   SPX 2020-01-28 expiry #0 1.0d: Generating extended strikes... removed 12 out of 56 strikes to ensure -1 < dC/dK[0] and dC/dK[-1] < 0: using 46 extended strikes based on 56 market strikes.
01:   SPX 2020-01-28 expiry #0 1.0d: Soft linear per-expiry no-arbitrage fit succeeded within bid/ask. This took 0.053s.
01:   SPX 2020-01-28 expiry #1 3.0d: Generating extended strikes... removed 5 out of 120 strikes to ensure -1 < dC/dK[0] and dC/dK[-1] < 0: using 117 extended strikes based on 120 market strikes.
01:   SPX 2020-01-28 expiry #1 3.0d: Could not hard fit with linear per-expiry no-arbitrage -- using masked soft fit instead which is outside bid/ask for 6 of 115 options. This took 0.13s.
01:   SPX 2020-01-28 expiry #1 3.0d: Could not hard fit with linear per-expiry no-arbitrage -- using masked soft fit instead which is outside bid/ask for 1 of 109 options. This took 0.11s.
01:   SPX 2020-01-28 expiry #1 3.0d: removing 11 of 120 

,spot,spot_prev,date,ticker,call_symbol,put_symbol,call_opt_id,put_opt_id,is_american,is_am,...,pure_fit1_gamma,pure_fit1_vega,pure_fit1_theta,fitX_vol,pure_fitX_price,pure_fitX_delta,pure_fitX_dk,pure_fitX_gamma,pure_fitX_vega,pure_fitX_theta
0,3276.23999,3243.629883,2020-01-28,SPX,SPXW 200129C3035000,SPXW 200129P3035000,131849080,131849198,False,False,...,0.000000,0.000000,0.000000,0.010000,0.074135,1.000000,-1.000000,0.000000,0.000000,0.000000
1,3276.23999,3243.629883,2020-01-28,SPX,SPXW 200129C3045000,SPXW 200129P3045000,131849082,131849200,False,False,...,0.000000,0.000000,0.000000,0.010000,0.071084,1.000000,-1.000000,0.000000,0.000000,0.000000
2,3276.23999,3243.629883,2020-01-28,SPX,SPXW 200129C3050000,SPXW 200129P3050000,131849083,131849201,False,False,...,0.000000,0.000000,0.000000,0.010000,0.069559,1.000000,-1.000000,0.000000,0.000000,0.000000
3,3276.23999,3243.629883,2020-01-28,SPX,SPXW 200129C3055000,SPXW 200129P3055000,131849084,131849202,False,False,...,0.000000,0.000000,0.000000,0.010000,0.068034,1.000000,-1.000000,0.000000,0.000000,0.000000
4,3276.23999,3243.629883,2020-01-28,SPX,SPXW 200129C3060000,SPXW 200129P3060000,131849085,131849203,False,False,...,0.056034,0.000071,0.002867,0.319933,0.066510,0.999693,-0.999669,0.056147,0.000071,0.002874
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5086,3276.23999,3243.629883,2020-01-28,SPX,SPX 210115C3650000,SPX 210115P3650000,131694745,131694747,False,True,...,2.365241,0.257827,0.014965,0.112489,0.010242,0.179468,-0.151945,2.365241,0.257827,0.014965
5087,3276.23999,3243.629883,2020-01-28,SPX,SPX 210115C3675000,SPX 210115P3675000,132061008,132061009,False,True,...,2.227982,0.239046,0.013656,0.110709,0.008708,0.159495,-0.134467,2.227961,0.239022,0.013654
5088,3276.23999,3243.629883,2020-01-28,SPX,SPX 210115C3700000,SPX 210115P3700000,131475177,131475271,False,True,...,2.082944,0.220502,0.012429,0.109253,0.007409,0.141344,-0.118633,2.083002,0.220530,0.012432
5089,3276.23999,3243.629883,2020-01-28,SPX,SPX 210115C3750000,SPX 210115P3750000,132029666,132029667,False,True,...,1.784483,0.184940,0.010205,0.106948,0.005345,0.109866,-0.091344,1.784483,0.184940,0.010205


## filter_options_day

,spot_prev,date,ticker,call_symbol,put_symbol,call_opt_id,put_opt_id,is_american,is_am,expiry,...,pure_fit1_theta,fitX_vol,pure_fitX_price,pure_fitX_delta,pure_fitX_dk,pure_fitX_gamma,pure_fitX_vega,pure_fitX_theta,spot_ret,spot_ref
48,0.990047,2020-01-28,SPX,SPXW 200129C3280000,SPXW 200129P3280000,131849129,131849247,False,False,2020-01-29,...,0.472707,0.149585,0.003465,0.476268,-0.472516,42.262302,0.025087,0.472823,0.010054,3276.23999
47,0.990047,2020-01-28,SPX,SPXW 200129C3275000,SPXW 200129P3275000,131849128,131849246,False,False,2020-01-29,...,0.489658,0.155358,0.004380,0.539401,-0.535514,40.564995,0.025008,0.489541,0.010054,3276.23999
49,0.990047,2020-01-28,SPX,SPXW 200129C3285000,SPXW 200129P3285000,131849130,131849248,False,False,2020-01-29,...,0.445990,0.144555,0.002670,0.409361,-0.405826,42.674805,0.024480,0.445868,0.010054,3276.23999
46,0.990047,2020-01-28,SPX,SPXW 200129C3270000,SPXW 200129P3270000,131849127,131849245,False,False,2020-01-29,...,0.491104,0.159938,0.005355,0.597943,-0.594041,38.397263,0.024370,0.491104,0.010054,3276.23999
50,0.990047,2020-01-28,SPX,SPXW 200129C3290000,SPXW 200129P3290000,131849131,131849249,False,False,2020-01-29,...,0.408551,0.140297,0.002001,0.341447,-0.338209,41.526264,0.023119,0.408686,0.010054,3276.23999
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5090,0.990047,2020-01-28,SPX,SPX 210115C3800000,SPX 210115P3800000,131475178,131475272,False,True,2021-01-15,...,0.008353,0.105600,0.003892,0.085085,-0.070024,1.498031,0.153296,0.008353,0.010054,3276.23999
2682,0.990047,2020-01-28,SPX,SPX 200320C3085000,SPX 200320P3085000,131373662,131373832,False,True,2020-03-20,...,0.069664,0.192220,0.066588,0.806146,-0.785507,3.770860,0.104123,0.069664,0.010054,3276.23999
1983,0.990047,2020-01-28,SPX,SPXW 200228C3120000,SPXW 200228P3120000,130538677,130538679,False,False,2020-02-28,...,0.084827,0.189138,0.053516,0.817108,-0.801924,4.742505,0.078308,0.084827,0.010054,3276.23999
1599,0.990047,2020-01-28,SPX,SPXW 200221C3135000,SPXW 200221P3135000,131046825,131046986,False,False,2020-02-21,...,0.095387,0.189892,0.048140,0.821601,-0.808440,5.290462,0.067772,0.095384,0.010054,3276.23999


In [2]:
print( raw_day['tte_y'].unique() )
print( raw_day['expiry'].unique() )
print( enr_day['tte_y'].unique() )
print( arb_free['tte_y'].unique() )
print( arb_free['expiry'].unique() )
print( opt_flt['tte_y'].unique() )

[0.00396825 0.01190476 0.01587302 0.02380952 0.03174603 0.03571429
 0.04365079 0.0515873  0.05555556 0.05952381 0.06746032 0.06428572
 0.07142857 0.07936508 0.08730159 0.09126984 0.09920635 0.10714286
 0.12698413 0.1468254  0.1436508  0.16666667 0.17460318 0.18650794
 0.22222222 0.21904762 0.2579365  0.3015873  0.29841268 0.33730158
 0.3968254  0.39365077 0.42460316 0.4690476  0.64365077 0.6785714
 0.72301584 0.82222223 0.897619   0.9325397  0.9690476  1.1396825
 1.3896825  1.8936509  2.8896825 ]
<DatetimeArray>
['2020-01-29 00:00:00', '2020-01-31 00:00:00', '2020-02-03 00:00:00',
 '2020-02-05 00:00:00', '2020-02-07 00:00:00', '2020-02-10 00:00:00',
 '2020-02-12 00:00:00', '2020-02-14 00:00:00', '2020-02-18 00:00:00',
 '2020-02-19 00:00:00', '2020-02-21 00:00:00', '2020-02-24 00:00:00',
 '2020-02-26 00:00:00', '2020-02-28 00:00:00', '2020-03-02 00:00:00',
 '2020-03-04 00:00:00', '2020-03-06 00:00:00', '2020-03-13 00:00:00',
 '2020-03-20 00:00:00', '2020-03-27 00:00:00', '2020-03-31 00:

In [3]:
arb_free.columns

Index(['spot', 'spot_prev', 'date', 'ticker', 'call_symbol', 'put_symbol',
       'call_opt_id', 'put_opt_id', 'is_american', 'is_am', 'expiry', 'strike',
       'fwd_ref', 'df', 'mkt_call_bid', 'mkt_put_bid', 'mkt_call_ask',
       'mkt_put_ask', 'mkt_call_log_volume', 'mkt_put_log_volume',
       'mkt_call_log_open_interest', 'mkt_put_log_open_interest', 'expiry_id',
       'bdte', 'tte_y', 'fwd', 'nms', 'nms_atm', 'call_ask', 'call_bid',
       'ask_vol', 'bid_vol', 'log_volume', 'log_open_interest', 'pure_strike',
       'pure_ask_price', 'pure_bid_price', 'pure_ask_delta', 'pure_bid_delta',
       'pure_ask_gamma', 'pure_bid_gamma', 'pure_bid_vega', 'pure_ask_vega',
       'pure_bid_theta', 'pure_ask_theta', 'pure_ask_dk', 'pure_bid_dk',
       'mask_arb_free', 'fit1_vol', 'pure_fit1_price', 'pure_fit1_delta',
       'pure_fit1_dk', 'pure_fit1_gamma', 'pure_fit1_vega', 'pure_fit1_theta',
       'fitX_vol', 'pure_fitX_price', 'pure_fitX_delta', 'pure_fitX_dk',
       'pure_fitX_gam